# ipynb import and export

> Reading and writing dialogs as Jupyter notebooks

In [ ]:
#| default_exp ipynb

`write_ipynb` converts a `Dialog` to a Jupyter notebook file or a JSON string. `read_ipynb` and `reads_ipynb` convert those forms back to dialogs. Other notebook tools can open the `.ipynb` files.

| Message type | Notebook cell type | Representation |
|-------------|--------------------|----------------|
| `note` | markdown | Message content becomes cell source |
| `prompt` | code | `solveit_ai` metadata, a `%%prompt` first line, and an `is_ai_res` output for the reply |
| `code` | code | Source and standard notebook outputs |
| `raw` | raw | Message content becomes cell source |

In [ ]:
#| export
from fastcore.utils import *
from fastcore.xtras import atomic_save
from fastcore.nbio import mk_cell,new_nb,dict2nb,read_nb,nb2str,repair_cell,repair_nb
import json
from contextlib import suppress
from base64 import b64encode,b64decode
from aidialog.dialog import *

In [ ]:
import random,os,nbformat
from nbformat.validator import NotebookValidationError
from tempfile import mkdtemp
from fastcore.test import *

In [ ]:
random.seed(7)
tstdir = Path(mkdtemp())
dlg = Dialog(name='dlg')
nt_msg = dlg.mk_message('A *test* dialog', msg_type=snote)
nt_msg.mk_attachment(b'not really a png', 'image/png')
code_msg = dlg.mk_message('1+1', msg_type=scode, output=code_output('2'))
ai_msg = dlg.mk_message('Add them.', msg_type=sprompt, output='The answer is **2**.')
raw_msg = dlg.mk_message('plain text', msg_type=sraw)
dlg


**dlg**


<details markdown='1'>

- A *test* dialog
- 1+1 ⇒ [{'output_type': 'execute_result', 'metadata': {}, 'data': …
- Add them. ⇒ [{'output_type': 'display_data', 'metadata': {'is_ai_res': …
- plain text

</details>

## Writing

Each attachment has an id and a MIME type. Binary data uses base64 in the notebook:

```json
{
  "attachments": {
    "image.png": {
      "image/png": "iVBO...kJggg=="
    }
  }
}
```

In [ ]:
#| export
def att2dict(att): return {att.content_type: b64encode(att.data).decode('ascii') if isinstance(att.data, bytes) else att.data}

Every message type supports attachments, but nbformat permits cell attachments only on markdown and raw cells. For code cells, aidialog follows rustygate's convention: store attachments in notebook metadata under `metadata.attachments[cell_id]`. Rustygate calls this location the attachment *home*.

`home_atts` moves code-cell attachments there before writing. `unhome_atts` restores them to their cells before reading. This matches rustygate's files and the ordinary cell representation its API returns. In memory, use `Message.attachments` regardless of the message type.

In [ ]:
#| export
def home_atts(nb):
    "Store code-cell attachments in notebook metadata using rustygate's convention"
    home = nb['metadata'].setdefault('attachments', {})
    for c in nb['cells']:
        if c['cell_type']=='code' and (atts := c.pop('attachments', None)): home[c['id']] = atts
    if not home: nb['metadata'].pop('attachments', None)
    return nb

def unhome_atts(nb):
    "Restore notebook-metadata attachments to their cells"
    home = nb['metadata'].pop('attachments', {})
    for c in nb['cells']:
        if (atts := home.get(c['id'])): c['attachments'] = atts
    return nb

A prompt contains both a question and an AI reply. `to_cell` writes it as a code cell with `solveit_ai: true` in metadata. The source starts with `%%prompt`, followed by the question. The reply uses the `display_data` output from `prompt_output`, with `is_ai_res` in its metadata.

Outside Solveit, a runner without the `%%prompt` magic raises `UsageError` rather than trying to execute the question as Python. A host can define that magic to run prompts. `cell2msg` removes the magic line when reconstructing the message.

In [ ]:
#| export
_out_meta_skip = {'__type'}

def _clean_out_meta(o):
    "A copy of output `o` without transient metadata; never mutates the live output"
    o = dict(o)
    if m := o.get('metadata'): o['metadata'] = {k:v for k,v in m.items() if k not in _out_meta_skip}
    return o

In [ ]:
#| export
@patch
def cell_meta(self:Message):
    "Build cell metadata from `meta` and declared attributes, omitting falsy attribute values"
    meta = dict(self.meta)
    for a,k in self.meta_attrs.items():
        if (v := getattr(self, a, None)): meta[k] = v
        else: meta.pop(k, None)
    return meta

`to_cell` calls `cell_meta` to prepare the cell's metadata. Override `cell_meta` when your host uses different types in memory and on disk. Call `super()` first, then convert the values it returns. For example, Solveit uses integer UI flags, but the notebook schema requires booleans for `collapsed` and `hide_input`.

In [ ]:
#| export
_prompt_magic = '%%prompt'

@patch
def to_cell(self:Message, version=2):
    "Convert message to a notebook cell"
    meta = self.cell_meta()
    src = self.source
    if self.msg_type==sprompt:
        meta['solveit_ai'] = True
        src = f'{_prompt_magic}\n{src}'
    outkw = {}
    if self.msg_type in (scode,sprompt) and self.output:
        outputs = self.output
        if version==1 and self.msg_type==scode: outputs = json.loads(outputs)
        outkw['outputs'] = [_clean_out_meta(o) for o in outputs]
    atts = {att.id: att2dict(att) for att in (self.attachments or [])}
    if atts: outkw['attachments'] = atts
    cell = mk_cell(src, self.cell_type, id=self.id, metadata=meta, **outkw)
    if repairs := repair_cell(cell): print('NB repair:', '; '.join(repairs))
    return cell

In [ ]:
om = Message('x', msg_type='code', output=[dict(output_type='execute_result', data={'text/plain':'42'}, metadata={'__type':'Safe'}, execution_count=1)])
test_eq(om.to_cell()['outputs'][0]['metadata'], {})  # transient metadata never reaches the file...
test_eq(om.output[0]['metadata'], {'__type':'Safe'})  # ...and serializing never mutates the live output (hosts read it after writes)

In [ ]:
test_eq(Message().to_cell()['metadata'],{})

`meta_attrs` maps attributes on a message to metadata keys in its notebook cell. `cell_meta` writes those attributes to their declared keys. It omits falsy attribute values. `cell2msg` restores the attributes when reading.

The `Message` constructor also extracts declared attributes from `meta=`. An attribute's default cannot hide a value supplied in that metadata. Keys outside `meta_attrs` remain in `Message.meta` and round-trip unchanged.

Here a host adds a `bookmark` attribute to its message class.

In [ ]:
class NoteMsg(Message): meta_attrs = dict(bookmark='bookmark')
class NoteDlg(Dialog): msg_cls = NoteMsg

bookmark_cell = NoteMsg(bookmark=9).to_cell()
test_eq(bookmark_cell['metadata']['bookmark'], 9)
test_eq(NoteMsg().to_cell()['metadata'], {})  # default/absent values aren't written

In [ ]:
test_eq(NoteMsg('x', meta=dict(bookmark=7)).bookmark, 7)  # constructor meta promotes like a file read
test_eq(NoteMsg('x', meta=dict(bookmark=7)).to_cell()['metadata'], {'bookmark': 7})  # ...so it round-trips to the file

This host accepts an integer `collapsed` flag in memory and writes it as a boolean. Its `cell_meta` override handles the conversion.

In [ ]:
class FlagMsg(Message):
    meta_attrs = dict(collapsed='collapsed')
    def cell_meta(self): return {k: bool(v) for k,v in super().cell_meta().items()}

assert FlagMsg(collapsed=1).to_cell()['metadata']['collapsed'] is True

In [ ]:
pr_msg = dlg.mk_message('What is 2+2?', output='The answer is 4.', msg_type='prompt')
pr_cell = pr_msg.to_cell()
test_eq((pr_cell['cell_type'], pr_cell['metadata']['solveit_ai']), ('code', True))
test_eq(pr_cell['source'], '%%prompt\nWhat is 2+2?')
test_eq(pr_cell['outputs'], prompt_output('The answer is 4.'))

pr_empty = dlg.mk_message('Hello?', output='', msg_type='prompt')
test_eq(pr_empty.to_cell()['outputs'], [])
pr_cell


```python
{ 'cell_type': 'code',
  'directives_': {},
  'execution_count': None,
  'id': 'fad71427',
  'idx_': 0,
  'lang_': 'python',
  'metadata': {'solveit_ai': True},
  'outputs': [ { 'data': {'text/markdown': 'The answer is 4.'},
                 'metadata': {'is_ai_res': True},
                 'output_type': 'display_data'}],
  'source': '%%prompt\nWhat is 2+2?'}
```

In [ ]:
#| export
def get_ipynb(dlg:Dialog, version=2, msgs=None):
    "Notebook object for `dlg`; `msgs` defaults to all its messages"
    cells = [m.to_cell(version=version) for m in (dlg.messages if msgs is None else msgs)]
    nb = new_nb(cells=cells, meta=dict(dlg.meta))
    home_atts(nb)
    if repairs := repair_nb(nb): print('NB repair:', '; '.join(repairs))
    return nb


In [ ]:
#| export
def safe_mtime(p):
    "mtime of `p`, its symlink if the target is missing, or None if `p` vanished (e.g. an atomic-save temp file)"
    with suppress(FileNotFoundError): return p.stat().st_mtime
    with suppress(FileNotFoundError): return p.stat(follow_symlinks=False).st_mtime

In [ ]:
#| export
def write_ipynb(dlg:Dialog, fname=None, version=2, msgs=None, **kwargs):
    """Write `dlg` as a notebook, or return its JSON string if `fname` is None.

    Pass `kwargs`, such as `uid` and `gid`, to `atomic_save`."""
    res = nb2str(get_ipynb(dlg, version=version, msgs=msgs))
    if not fname: return res
    fname = Path(fname).expanduser()
    with atomic_save(fname, mode='w', encoding='utf-8', **kwargs) as f: f.write(res)
    dlg.mtime_ = safe_mtime(fname)

In [ ]:
write_ipynb(dlg, fname=tstdir/'dlg.ipynb', version=2)

In [ ]:
#| export
@patch
def write(self:Dialog, base_path, version=2, msgs=None, **kwargs):
    write_ipynb(self, Path(base_path).expanduser()/f'{self.name}.ipynb', version=version, msgs=msgs, **kwargs)

In [ ]:
dlg.write(tstdir)

In [ ]:
#| export
def ipynb_cells(path, nm, prefix=None, suffix=None):
    tmpl = Path(path).expanduser()/f'{nm}.ipynb'
    if not tmpl.exists(): return []
    try: nb = read_nb(tmpl)
    except json.JSONDecodeError: return []
    if repairs := repair_nb(nb): print('NB repair:', '; '.join(repairs))
    return listify(prefix) + nb.cells + listify(suffix)

In [ ]:
test_eq(len(ipynb_cells(tstdir, dlg.name)), len(dlg.messages))
ipynb_cells(tstdir, dlg.name)[0]

```python
{ 'attachments': { '2530bb1d-6d13-4cde-8623-7b2ed91e3f72': { 'image/png': 'bm90IHJlYWxseSBhIHBuZw=='}},
  'cell_type': 'markdown',
  'id': 'a54dca18',
  'idx_': 0,
  'lang_': 'python',
  'metadata': {},
  'source': 'A *test* dialog'}
```

## Reading

In [ ]:
#| export
def dict2att(att_id, att_data):
    "Convert attachment dict to Attachment object"
    content_type, data = first(att_data.items())
    if isinstance(data, str): data = b64decode(data)
    return Attachment(data, content_type, att_id)

def _output_from_cell(cell):
    if cell.cell_type!='code': return ''
    return getattr(cell, 'outputs', [])

In [ ]:
#| export
@patch
def cell2msg(self:Dialog, cell):
    "Convert single notebook cell to message object"
    meta = dict(cell.metadata)
    kwargs = {a: meta.pop(k) for a,k in self.msg_cls.meta_attrs.items() if k in meta}
    content = cell.get('source', '')
    msg_type = sprompt if meta.pop('solveit_ai', 0) else snote if cell.cell_type=='markdown' else cell.cell_type
    if msg_type==sprompt and content.split('\n', 1)[0].strip()==_prompt_magic:
        content = content.split('\n', 1)[1] if '\n' in content else ''
    output = '' if msg_type in (snote,sraw) else _output_from_cell(cell)
    atts = [dict2att(att_id, att_data) for att_id, att_data in cell.get('attachments', {}).items()]
    id = getattr(cell, 'id', None) or rtoken_hex(4)
    return self.msg_cls(content, id=id, output=output, msg_type=msg_type, dlg=self, attachments=atts, meta=meta, **kwargs)

In [ ]:
back = NoteDlg(name='t').cell2msg(bookmark_cell)
test_eq(back.bookmark, 9)
test_eq(back.meta, {})  # promoted out of `meta` into the attribute

In [ ]:
# Test roundtrip prompts
pr_back = dlg.cell2msg(pr_cell)
test_eq(pr_back.content, 'What is 2+2?')
test_eq(pr_back.msg_type, 'prompt')
test_eq(pr_back.ai_res, 'The answer is 4.')

In [ ]:
#| export
@patch
def from_cells(self:Dialog, cells):
    self.messages = Msgs(cells).map(self.cell2msg)
    return self

In [ ]:
#| export
def reads_ipynb(txt, cls=Dialog, name='dialog', verbose=False):
    "Read a dialog from notebook JSON string `txt`, constructing via `cls`"
    nb = json.loads(txt)
    if (repairs := repair_nb(nb)) and verbose: print('NB repair:', '; '.join(repairs))
    unhome_atts(nb)
    nb = dict2nb(nb)
    return cls(name=name, meta=dict(nb.get('metadata', {}))).from_cells(nb.cells)

In [ ]:
#| export
def read_ipynb(fname, cls=Dialog, name=None, verbose=False):
    """Read notebook file `fname` into a dialog of type `cls`.

    Set the filename's suffix to `.ipynb`. Default `name` to the file stem."""
    f = Path(fname).expanduser()
    if f.suffix != '.ipynb': f = f.with_suffix('.ipynb')
    if not f.exists(): return print(f,'does not exist')
    try: res = reads_ipynb(f.read_text(encoding='utf-8'), cls, name or f.stem, verbose=verbose)
    except (json.JSONDecodeError, PermissionError): return
    res.path_ = f
    res.mtime_ = safe_mtime(f)
    return res

In [ ]:
dlg = read_ipynb(tstdir/'dlg')
dlg


**dlg**


<details markdown='1'>

- A *test* dialog
- 1+1 ⇒ [{'data': {'text/plain': '2'}, 'execution_count': 1, 'metad…
- Add them. ⇒ [{'data': {'text/markdown': 'The answer is **2**.'}, 'metad…
- plain text
- What is 2+2? ⇒ [{'data': {'text/markdown': 'The answer is 4.'}, 'metadata'…
- Hello?

</details>

In [ ]:
s = (tstdir/'dlg.ipynb').read_text()
dlg2 = reads_ipynb(s)
test_eq(dlg2.name, 'dialog')
test_eq(write_ipynb(dlg2), s)
with expect_fail(json.JSONDecodeError): reads_ipynb('not json')

This example writes a code message with an image attachment. The file passes nbformat validation because the attachment uses `metadata.attachments`, keyed by cell id. Reading restores the attachment's original bytes to the message. Writing the reconstructed dialog produces the same JSON.

In [ ]:
hm = Dialog(name='home')
cm = hm.mk_message('1+1', msg_type=scode, output=code_output('2'))
cm.mk_attachment(b'not really a png', 'image/png')
hs = write_ipynb(hm)
nbformat.validate(nbformat.reads(hs, as_version=4))
hnb = json.loads(hs)
assert 'attachments' not in hnb['cells'][0]
test_eq(reads_ipynb(hs).messages[0].attachments[0].data, b'not really a png')
test_eq(write_ipynb(reads_ipynb(hs)), hs)
hnb['metadata']['attachments']

{'e9232f8a': {'f2211f9e-e491-45b1-abec-b5563bfc1e6f': {'image/png': 'bm90IHJlYWxseSBhIHBuZw=='}}}

In [ ]:
#| export
@patch
def save(self:Dialog, fname=None):
    "Write back to `fname`, or to the `path_` stamped by `read_ipynb`"
    fname = fname or self.path_
    if not fname: raise ValueError('no fname passed, and no `path_` stamped by read_ipynb')
    write_ipynb(self, fname)

`read_ipynb` records the source file in `dlg.path_`. You can edit the dialog and call `save()` without passing the path again. The trailing underscore distinguishes this working attribute from a host's own `path` property.

`read_ipynb` and file writes through `write_ipynb` record the file's `safe_mtime` in `dlg.mtime_`. A host can compare that timestamp with the current file timestamp when checking for external writes. A new dialog has `None` for both attributes. `save()` needs an explicit filename until the dialog has a `path_`.

The file below already uses aidialog's normalized format. Saving it without edits preserves its bytes, although the save still updates the file's timestamp.

In [ ]:
before = (tstdir/'dlg.ipynb').read_text()
dlg.save()
test_eq((tstdir/'dlg.ipynb').read_text(), before)
test_eq(dlg.mtime_, dlg.path_.stat().st_mtime)
assert isinstance(dlg.messages, Msgs)
unread = Dialog(name='unread')
test_eq((unread.path_, unread.mtime_), (None,None))
with expect_fail(ValueError, 'path_'): unread.save()

### Metadata preservation

The base `Message` class declares two metadata attributes. `skipped` hides a message from AI context. `pinned` keeps it through context eviction. Both use Solveit's literal metadata keys for compatibility between hosts.

Other application metadata remains in `Message.meta` unless a subclass declares it in `meta_attrs`. Notebook-level application metadata remains in `Dialog.meta`. The plain classes preserve those annotations when reading and writing files from another host.

In [ ]:
m0 = dlg.messages[0]
m0.meta['my_app_flag'] = dict(level=3)
dlg.meta['my_app'] = dict(version=1)
dlg.write(tstdir)
dlg = read_ipynb(tstdir/'dlg')
test_eq(dlg.messages[0].meta['my_app_flag'], dict(level=3))
test_eq(dlg.meta['my_app'], dict(version=1))

Set `skipped` and `pinned` as message attributes. Saving writes truthy values to their metadata keys. Reading restores them as attributes and removes those keys from `Message.meta`. Unset or falsy attributes add no metadata to the file.

In [ ]:
m0 = dlg.messages[0]
m0.skipped = 1
m0.pinned = True
dlg.write(tstdir)
rt = read_ipynb(tstdir/'dlg')
test_eq(rt.messages[0].skipped, 1)                  # promoted back as a real attribute
test_eq(rt.messages[0].pinned, True)
assert 'skipped' not in rt.messages[0].meta         # claimed by meta_attrs, so not left in meta
test_eq(rt.messages[1].skipped, 0)                  # untouched message: class default, and
assert 'skipped' not in rt.messages[1].cell_meta()  # ...falsy means nothing written to the file

### Round-trip fidelity

`write_ipynb` normalizes notebooks. For example, it writes `null` for each code cell's execution count, even if that cell has run. The first save can therefore change the input file without changing its source or displayed output.

After normalization, reading and writing an unedited dialog produces identical JSON with this serializer. Changes in that JSON indicate changes to the serialized dialog. Other notebook tools can still change formatting or serialization without changing the message content.

We check this property against the repository's notebooks as well as the constructed examples.

In [ ]:
for p in sorted(Path('.').glob('0*.ipynb')):
    s = write_ipynb(read_ipynb(str(p)))
    test_eq(write_ipynb(reads_ipynb(s)), s)

Hand-edited notebooks can contain fields that nbformat rejects. This example reproduces a real file's stray `outputs` key on a markdown cell. `read_ipynb` calls `repair_nb` before constructing messages. The repaired notebook keeps the source text and omits the invalid key. Writing it produces a file that passes validation.

In [ ]:
bad_cells = [dict(cell_type='markdown', id='aaaa1111', metadata={}, source='hi', outputs=[])]
(tstdir/'bad.ipynb').write_text(json.dumps(dict(nbformat=4, nbformat_minor=5, metadata={}, cells=bad_cells)))
with expect_fail(NotebookValidationError): nbformat.validate(nbformat.read(tstdir/'bad.ipynb', as_version=4))
healed = write_ipynb(read_ipynb(tstdir/'bad'))
nbformat.validate(nbformat.reads(healed, as_version=4))
assert 'hi' in healed

## Converting old prompts

`conv_old_prompts` converts old markdown prompt cells to the current code-cell format. It changes the supplied notebook object in place and returns the changed cell ids. A host can use that list to decide whether to write the file back after opening it.

A second call leaves the converted cells unchanged and returns an empty list. After converting any prompts, `conv_old_prompts` calls `home_atts` to move their attachments into notebook metadata. Code cells cannot store those attachments directly.

In [ ]:
#| export
_reply_sep = "\n\n##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\n\n"

def conv_old_prompts(nb):
    "Convert old markdown prompt cells to code cells in place and return their changed ids"
    changed = []
    for c in nb['cells']:
        if c['cell_type']!='markdown' or not c['metadata'].get('solveit_ai'): continue
        if isinstance(c['source'], list): c['source'] = ''.join(c['source'])
        content,*reply = c['source'].split(_reply_sep)
        c['cell_type'] = 'code'
        c['source'] = f'{_prompt_magic}\n{content}'
        c['outputs'] = prompt_output(reply[0]) if reply else []
        c['execution_count'] = None
        changed.append(c['id'])
    if changed: home_atts(nb)
    return changed

Old prompt cells use markdown with `solveit_ai` metadata. Their source contains the question and reply separated by `_reply_sep`. Current writers use code cells instead.

The example builds an old prompt using that separator. Conversion preserves its question, reply, and attachment. The new notebook passes schema validation and reconstructs the same message when read.

In [ ]:
onb = dict(nbformat=4, nbformat_minor=5, metadata={}, cells=[dict(
    cell_type='markdown', id='opr1', metadata=dict(solveit_ai=True),
    source='What is 7*6?' + _reply_sep + '**42**.',
    attachments={'att1': {'image/png': 'bm90IHJlYWxseSBhIHBuZw=='}})])
test_eq(conv_old_prompts(onb), ['opr1'])
ocell = onb['cells'][0]
test_eq((ocell['cell_type'], ocell['source']), ('code', '%%prompt\nWhat is 7*6?'))
assert 'attachments' not in ocell                # the retype homed the attachment
nbformat.validate(nbformat.from_dict(onb))
test_eq(conv_old_prompts(onb), [])               # converted cells no longer match: idempotent
back = reads_ipynb(json.dumps(onb)).messages[0]
test_eq((back.content, back.ai_res), ('What is 7*6?', '**42**.'))
test_eq(back.attachments[0].data, b'not really a png')
back.ai_res

'**42**.'

## export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()